In [1]:
!pip -q install transformers datasets jiwer sentencepiece accelerate librosa soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 75.5 MB/s eta 0:00:00


In [2]:
import os
import json
import random
import re
import numpy as np
import pandas as pd
import torch
import librosa

from tqdm.auto import tqdm
from jiwer import wer, cer

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModel
)
import torch.nn as nn

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ===============================
# INPUT DATASET
# ===============================
INPUT_CSV = "/content/drive/MyDrive/dataset_corrected_juniors.csv"

# sampled validation csv
SAMPLED_CSV = "/content/drive/MyDrive/dataset_corrected_juniors_sample_1000.csv"

# ===============================
# WHISPER MODEL
# ===============================
MERGED_MODEL_PATH = "/content/drive/MyDrive/vassista22_code_switching_adalora_checkpoints_phase1_experiment4/merged_full_model"

# ===============================
# XLM-R LARGE MODEL
# ===============================
XLMR_OUTPUT_DIR = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2"
XLMR_MODEL_DIR = os.path.join(XLMR_OUTPUT_DIR, "best_model")

# label maps from training
ROOT_MAP_PATH = os.path.join(XLMR_OUTPUT_DIR, "id2root.json")
SUFFIX_MAP_PATH = os.path.join(XLMR_OUTPUT_DIR, "id2suffix.json")

# ===============================
# OUTPUT FILES
# ===============================
ASR_RESULTS_CSV = "/content/drive/MyDrive/asr_results_sample_1000.csv"
XLMR_RESULTS_CSV = "/content/drive/MyDrive/asr_xlmr_results_sample_1000.csv"
METRICS_JSON = "/content/drive/MyDrive/asr_xlmr_metrics_sample_1000.json"

# audio
TARGET_SR = 16000

# sampling
SAMPLE_SIZE = 1000
RANDOM_SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [5]:
df = pd.read_csv(INPUT_CSV)

print("Columns:", df.columns.tolist())
print("Total rows:", len(df))

required_cols = ["id", "script_text", "audio_wav_path", "duration"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)

sample_df.to_csv(SAMPLED_CSV, index=False)

print("Sampled rows:", len(sample_df))
print("Saved sampled CSV to:", SAMPLED_CSV)
sample_df.head()

Columns: ['id', 'script_text', 'audio_wav_path', 'duration']
Total rows: 25897
Sampled rows: 1000
Saved sampled CSV to: /content/drive/MyDrive/dataset_corrected_juniors_sample_1000.csv


,id,script_text,audio_wav_path,duration
0,8a13434a-3693-40d9-986a-a231316ecda2,Online class join பண்ண late ஆனதால start miss ஆ...,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,"ரெண்டு table book பண்ணுங்க please, ஜன்னல் பக்க...",/content/drive/MyDrive/audio_wav_final/audio-r...,4.14
2,fd532190-fb3d-4bce-9139-c1cc6a644601,Etsy Help Center site-ல Selling on Etsy-ல Paym...,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86


In [6]:
processor = WhisperProcessor.from_pretrained(MERGED_MODEL_PATH)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    MERGED_MODEL_PATH,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

if device != "cuda":
    whisper_model = whisper_model.to(device)

whisper_model.eval()

print("Whisper model loaded.")

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Whisper model loaded.


In [7]:
def load_audio(audio_path, target_sr=16000):
    audio, sr = librosa.load(audio_path, sr=target_sr)
    return audio

In [19]:
import re

def clean_whisper_text(text):
    text = re.sub(r"<\|.*?\|>", "", text)   # remove special tokens like <|startoftranscript|>
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [22]:
from transformers.utils import logging
logging.set_verbosity_error()

whisper_model.generation_config.max_length = None
def transcribe_audio_whisper(audio_path):
    try:
        audio = load_audio(audio_path, TARGET_SR)

        inputs = processor(
            audio,
            sampling_rate=TARGET_SR,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16 if device == "cuda" else torch.float32
        )

        with torch.no_grad():
            predicted_ids = whisper_model.generate(
                input_features,
                max_new_tokens=225
            )

        text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        text = clean_whisper_text(text)

        return text.strip(), None

    except Exception as e:
        return "", str(e)

In [21]:
from transformers.utils import logging
logging.set_verbosity_error()

whisper_model.generation_config.max_length = None

def transcribe_audio_whisper(audio_path):
    try:
        audio = load_audio(audio_path, TARGET_SR)

        inputs = processor(
            audio,
            sampling_rate=TARGET_SR,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(
            device=device,
            dtype=torch.float16 if device == "cuda" else torch.float32
        )

        with torch.no_grad():
            predicted_ids = whisper_model.generate(
                input_features,
                max_new_tokens=225
            )

        text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        return text.strip(), None

    except Exception as e:
        return "", str(e)

In [23]:
asr_rows = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Running Whisper ASR"):
    row_id = row["id"]
    expected_text = str(row["script_text"]).strip()
    audio_path = str(row["audio_wav_path"]).strip()
    duration = row["duration"]

    generated_text, error_msg = transcribe_audio_whisper(audio_path)

    asr_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "error": error_msg
    })

asr_df = pd.DataFrame(asr_rows)
asr_df.to_csv(ASR_RESULTS_CSV, index=False)

print("Saved ASR results to:", ASR_RESULTS_CSV)
asr_df.head()

Running Whisper ASR:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved ASR results to: /content/drive/MyDrive/asr_results_sample_1000.csv


,id,audio_wav_path,duration,expected_text,generated_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,"ரெண்டு table book பண்ணுங்க please, ஜன்னல் பக்க...",Render table book பண்ணுங்க please. ஜன்னல் பக்க...,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy Help Center site-ல Selling on Etsy-ல Paym...,ITZY help center site-ல selling on ITZY-ல paym...,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,None


In [24]:
valid_asr_df = asr_df[asr_df["error"].isna() | (asr_df["error"] == "")].copy()

expected_list = valid_asr_df["expected_text"].astype(str).tolist()
generated_list = valid_asr_df["generated_text"].astype(str).tolist()

asr_wer = wer(expected_list, generated_list)
asr_cer = cer(expected_list, generated_list)

print("ASR WER:", asr_wer)
print("ASR CER:", asr_cer)
print("Valid rows:", len(valid_asr_df))

ASR WER: 0.2708534233365477
ASR CER: 0.07405905804446246
Valid rows: 1000


In [25]:
def clean_token(tok):
    tok = str(tok).strip()
    tok = tok.strip(".,!?;:\"“”‘’()[]{}")
    return tok

def tokenize(text):
    return [clean_token(tok) for tok in str(text).strip().split() if clean_token(tok)]

def is_english_word(token):
    return bool(re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", token))

def is_tamil_text(text):
    return bool(re.search(r"[\u0B80-\u0BFF]", text))

def is_mixed_token(token):
    if "-" not in token:
        return False
    parts = token.split("-", 1)
    if len(parts) != 2:
        return False
    left, right = parts[0].strip(), parts[1].strip()
    return is_english_word(left) and is_tamil_text(right)

def get_token_class(token):
    if is_mixed_token(token):
        return "MIX"
    elif is_english_word(token):
        return "EN"
    elif is_tamil_text(token):
        return "TA"
    else:
        return "OTHER"

def split_mixed_token(token):
    if "-" not in token:
        return None, None
    left, right = token.split("-", 1)
    return left.strip(), right.strip()

def extract_root_suffix(token, token_class):
    if token_class == "MIX":
        root, suffix = split_mixed_token(token)
        return root if root else "", suffix if suffix else ""
    elif token_class == "EN":
        return token, "NULL"
    else:
        return "", ""

def build_model_input(left_context, token, right_context, token_class, root, suffix):
    return f"LEFT={left_context} TOKEN={token} RIGHT={right_context} CLASS={token_class} ROOT={root} SUFFIX={suffix}"

def get_context(tokens, idx, window=2):
    left_tokens = tokens[max(0, idx-window):idx]
    right_tokens = tokens[idx+1:idx+1+window]
    return " ".join(left_tokens).strip(), " ".join(right_tokens).strip()

In [26]:
with open(ROOT_MAP_PATH, "r", encoding="utf-8") as f:
    id2root = json.load(f)

with open(SUFFIX_MAP_PATH, "r", encoding="utf-8") as f:
    id2suffix = json.load(f)

# keys may be strings in json
id2root = {int(k): v for k, v in id2root.items()}
id2suffix = {int(k): v for k, v in id2suffix.items()}

XLMR_MODEL_NAME = "xlm-roberta-large"
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_MODEL_NAME)

print("Loaded label maps.")
print("Root labels:", len(id2root))
print("Suffix labels:", len(id2suffix))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded label maps.
Root labels: 794
Suffix labels: 62


In [27]:
class XLMRDualHeadModel(nn.Module):
    def __init__(self, model_name, num_root_labels, num_suffix_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.root_classifier = nn.Linear(hidden_size, num_root_labels)
        self.suffix_classifier = nn.Linear(hidden_size, num_suffix_labels)

    def forward(self, input_ids=None, attention_mask=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]
        cls_repr = self.dropout(cls_repr)

        root_logits = self.root_classifier(cls_repr)
        suffix_logits = self.suffix_classifier(cls_repr)

        return {
            "root_logits": root_logits,
            "suffix_logits": suffix_logits
        }

xlmr_model = XLMRDualHeadModel(
    model_name=XLMR_MODEL_NAME,
    num_root_labels=len(id2root),
    num_suffix_labels=len(id2suffix)
)

state_dict_path = os.path.join(XLMR_MODEL_DIR, "model.safetensors")

# If safetensors exists
if os.path.exists(state_dict_path):
    from safetensors.torch import load_file
    state_dict = load_file(state_dict_path)
    xlmr_model.load_state_dict(state_dict)
else:
    # fallback if pytorch_model.bin exists
    state_dict_path = os.path.join(XLMR_MODEL_DIR, "pytorch_model.bin")
    state_dict = torch.load(state_dict_path, map_location="cpu")
    xlmr_model.load_state_dict(state_dict)

xlmr_model.to(device)
xlmr_model.eval()

print("Loaded trained XLM-R large dual-head model.")

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded trained XLM-R large dual-head model.


In [28]:
MAX_LEN = 96

def predict_corrected_token(tokens, idx):
    token = tokens[idx]
    token_class = get_token_class(token)

    if token_class not in ["EN", "MIX"]:
        return token, None, None

    wrong_root, wrong_suffix = extract_root_suffix(token, token_class)

    left_context, right_context = get_context(tokens, idx, window=2)

    model_input = build_model_input(
        left_context=left_context,
        token=token,
        right_context=right_context,
        token_class=token_class,
        root=wrong_root,
        suffix=wrong_suffix
    )

    enc = xlmr_tokenizer(
        model_input,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = xlmr_model(input_ids=input_ids, attention_mask=attention_mask)
        root_pred_id = torch.argmax(outputs["root_logits"], dim=1).item()
        suffix_pred_id = torch.argmax(outputs["suffix_logits"], dim=1).item()

    pred_root = id2root[root_pred_id]
    pred_suffix = id2suffix[suffix_pred_id]

    if token_class == "EN":
        corrected_token = pred_root
    else:
        corrected_token = f"{pred_root}-{pred_suffix}"

    return corrected_token, pred_root, pred_suffix

In [29]:
def correct_sentence_with_xlmr(asr_sentence):
    tokens = tokenize(asr_sentence)
    corrected_tokens = []
    token_debug = []

    for idx in range(len(tokens)):
        original_token = tokens[idx]
        token_class = get_token_class(original_token)

        if token_class in ["EN", "MIX"]:
            corrected_token, pred_root, pred_suffix = predict_corrected_token(tokens, idx)
        else:
            corrected_token = original_token
            pred_root = None
            pred_suffix = None

        corrected_tokens.append(corrected_token)

        token_debug.append({
            "index": idx,
            "original_token": original_token,
            "token_class": token_class,
            "corrected_token": corrected_token,
            "pred_root": pred_root,
            "pred_suffix": pred_suffix
        })

    corrected_sentence = " ".join(corrected_tokens)
    return corrected_sentence, token_debug

In [30]:
xlmr_rows = []

for idx, row in tqdm(asr_df.iterrows(), total=len(asr_df), desc="Running XLM-R correction"):
    row_id = row["id"]
    expected_text = str(row["expected_text"]).strip()
    generated_text = str(row["generated_text"]).strip()
    audio_path = str(row["audio_wav_path"]).strip()
    duration = row["duration"]
    error_msg = row["error"]

    if error_msg is not None and str(error_msg).strip() != "":
        xlmr_corrected_text = ""
        token_debug = []
    else:
        xlmr_corrected_text, token_debug = correct_sentence_with_xlmr(generated_text)

    xlmr_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "xlmr_corrected_text": xlmr_corrected_text,
        "error": error_msg
    })

xlmr_df = pd.DataFrame(xlmr_rows)
xlmr_df.to_csv(XLMR_RESULTS_CSV, index=False)

print("Saved XLM-R results to:", XLMR_RESULTS_CSV)
xlmr_df.head()

Running XLM-R correction:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved XLM-R results to: /content/drive/MyDrive/asr_xlmr_results_sample_1000.csv


,id,audio_wav_path,duration,expected_text,generated_text,xlmr_corrected_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,online class India பண்ண late ஆனதால late miss ஆ...,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,"ரெண்டு table book பண்ணுங்க please, ஜன்னல் பக்க...",Render table book பண்ணுங்க please. ஜன்னல் பக்க...,table table book பண்ணுங்க pay ஜன்னல் பக்கமா பண...,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy Help Center site-ல Selling on Etsy-ல Paym...,ITZY help center site-ல selling on ITZY-ல paym...,Etsy help Centre site-ல site site Etsy-ல Etsy ...,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,None


In [31]:
valid_xlmr_df = xlmr_df[xlmr_df["error"].isna() | (xlmr_df["error"] == "")].copy()

expected_list = valid_xlmr_df["expected_text"].astype(str).tolist()
generated_list = valid_xlmr_df["generated_text"].astype(str).tolist()
xlmr_generated_list = valid_xlmr_df["xlmr_corrected_text"].astype(str).tolist()

metrics = {
    "num_rows_total": int(len(xlmr_df)),
    "num_rows_valid": int(len(valid_xlmr_df)),
    "asr_wer": float(wer(expected_list, generated_list)),
    "asr_cer": float(cer(expected_list, generated_list)),
    "xlmr_wer": float(wer(expected_list, xlmr_generated_list)),
    "xlmr_cer": float(cer(expected_list, xlmr_generated_list)),
}

print(json.dumps(metrics, indent=2, ensure_ascii=False))

with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Saved metrics to:", METRICS_JSON)

{
  "num_rows_total": 1000,
  "num_rows_valid": 1000,
  "asr_wer": 0.2708534233365477,
  "asr_cer": 0.07405905804446246,
  "xlmr_wer": 0.44021215043394407,
  "xlmr_cer": 0.21415298330968308
}
Saved metrics to: /content/drive/MyDrive/asr_xlmr_metrics_sample_1000.json


In [32]:
show_df = xlmr_df[["expected_text", "generated_text", "xlmr_corrected_text"]].head(20)
show_df

,expected_text,generated_text,xlmr_corrected_text
0,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,online class India பண்ண late ஆனதால late miss ஆ...
1,"ரெண்டு table book பண்ணுங்க please, ஜன்னல் பக்க...",Render table book பண்ணுங்க please. ஜன்னல் பக்க...,table table book பண்ணுங்க pay ஜன்னல் பக்கமா பண...
2,Etsy Help Center site-ல Selling on Etsy-ல Paym...,ITZY help center site-ல selling on ITZY-ல paym...,Etsy help Centre site-ல site site Etsy-ல Etsy ...
3,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...
4,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...
5,Tubelight சரியா எரியேல ஒருக்கா repair செய்யணும்,Tube light சரியா எரியேல ஒருக்கா repair செய்யணும்,light light சரியா எரியேல ஒருக்கா repair செய்யணும்
6,பிழை விட்ட questions-அ அவர்ட்ட கேட்டா explain ...,பிளவுட்ட questions-அ அவர்கிட்ட கேட்டா explain ...,பிளவுட்ட questions-அ அவர்கிட்ட கேட்டா XM பண்ணு...
7,என்ர cycle puncture ஆயிட்டு அதான் repair-க்கு ...,என்ட cycle puncture ஆயிட்டு அதை repair-க்கு கொ...,என்ட cycle torch ஆயிட்டு அதை repair-க்கு கொண்ட...
8,"Office system login problem இருந்ததால, IT supp...","Office system login problem இருந்ததால, IT supp...",Office India problem problem இருந்ததால support...
9,இந்த chocolate செம்ம taste-ஆ இருக்கும் வாங்குவம்.,இந்த chocolate செம taste-ஆ இருக்கும் வாங்குவம்,இந்த spicy செம taste-ஆ இருக்கும் வாங்குவம்


# Correction 1

In [33]:
import pandas as pd
from collections import Counter

TRAIN_DATA_PATH = "/content/drive/MyDrive/dual_head_training_data.csv"

train_df = pd.read_csv(TRAIN_DATA_PATH)

# only rows where generated token != expected token
changed_df = train_df[train_df["generated_token"] != train_df["expected_token"]].copy()

token_change_counter = Counter(changed_df["generated_token"].astype(str).tolist())

# tokens that appeared at least 2 times as wrong
SUSPICIOUS_TOKEN_MIN_COUNT = 2
suspicious_tokens = {
    tok for tok, cnt in token_change_counter.items()
    if cnt >= SUSPICIOUS_TOKEN_MIN_COUNT
}

print("Total changed rows:", len(changed_df))
print("Unique suspicious tokens:", len(suspicious_tokens))

# Optional: save for inspection
pd.DataFrame(sorted(list(suspicious_tokens)), columns=["suspicious_token"]).to_csv(
    "/content/drive/MyDrive/suspicious_tokens_for_xlmr.csv", index=False
)

Total changed rows: 28658
Unique suspicious tokens: 2838


In [34]:
import torch
import torch.nn.functional as F

MAX_LEN = 96

def predict_token_with_confidence(tokens, idx):
    token = tokens[idx]
    token_class = get_token_class(token)

    if token_class not in ["EN", "MIX"]:
        return {
            "original_token": token,
            "token_class": token_class,
            "pred_root": None,
            "pred_suffix": None,
            "root_conf": None,
            "suffix_conf": None,
            "corrected_token": token
        }

    wrong_root, wrong_suffix = extract_root_suffix(token, token_class)
    left_context, right_context = get_context(tokens, idx, window=2)

    model_input = build_model_input(
        left_context=left_context,
        token=token,
        right_context=right_context,
        token_class=token_class,
        root=wrong_root,
        suffix=wrong_suffix
    )

    enc = xlmr_tokenizer(
        model_input,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = xlmr_model(input_ids=input_ids, attention_mask=attention_mask)

        root_probs = F.softmax(outputs["root_logits"], dim=1)
        suffix_probs = F.softmax(outputs["suffix_logits"], dim=1)

        root_pred_id = torch.argmax(root_probs, dim=1).item()
        suffix_pred_id = torch.argmax(suffix_probs, dim=1).item()

        root_conf = root_probs[0, root_pred_id].item()
        suffix_conf = suffix_probs[0, suffix_pred_id].item()

    pred_root = id2root[root_pred_id]
    pred_suffix = id2suffix[suffix_pred_id]

    if token_class == "EN":
        corrected_token = pred_root
    else:
        corrected_token = f"{pred_root}-{pred_suffix}"

    return {
        "original_token": token,
        "token_class": token_class,
        "wrong_root": wrong_root,
        "wrong_suffix": wrong_suffix,
        "pred_root": pred_root,
        "pred_suffix": pred_suffix,
        "root_conf": root_conf,
        "suffix_conf": suffix_conf,
        "corrected_token": corrected_token
    }

In [77]:
ROOT_CONF_THRESH_EN = 0.95
ROOT_CONF_THRESH_MIX = 0.3
SUFFIX_CONF_THRESH_MIX = 0.3

In [78]:
def should_correct_token(pred_info, suspicious_tokens):
    token_class = pred_info["token_class"]
    original_token = pred_info["original_token"]
    corrected_token = pred_info["corrected_token"]

    # no correction needed for non EN/MIX
    if token_class not in ["EN", "MIX"]:
        return False

    # if prediction did not change anything, keep original
    if corrected_token == original_token:
        return False

    # suspicious token check
    is_suspicious = original_token in suspicious_tokens

    if token_class == "EN":
        # For EN tokens:
        # change only if:
        # 1. high confidence, and
        # 2. either suspicious token OR extremely high confidence
        if pred_info["root_conf"] >= ROOT_CONF_THRESH_EN and (is_suspicious or pred_info["root_conf"] >= 0.98):
            return True
        return False

    if token_class == "MIX":
        # For MIX tokens:
        # change only if:
        # 1. root confidence high
        # 2. suffix confidence high
        # 3. token suspicious OR very high confidence
        if (
            pred_info["root_conf"] >= ROOT_CONF_THRESH_MIX and
            pred_info["suffix_conf"] >= SUFFIX_CONF_THRESH_MIX and
            (is_suspicious or pred_info["root_conf"] >= 0.97)
        ):
            return True
        return False

    return False

In [73]:
def should_correct_token(pred_info, suspicious_tokens):
    token_class = pred_info["token_class"]
    original_token = pred_info["original_token"]
    corrected_token = pred_info["corrected_token"]

    if token_class != "MIX":
        return False

    if corrected_token == original_token:
        return False

    is_suspicious = original_token in suspicious_tokens

    if (
        pred_info["root_conf"] >= ROOT_CONF_THRESH_MIX and
        pred_info["suffix_conf"] >= SUFFIX_CONF_THRESH_MIX and
        (is_suspicious or pred_info["root_conf"] >= 0.97)
    ):
        return True

    return False

In [79]:
def correct_sentence_with_xlmr_safe(asr_sentence, suspicious_tokens):
    tokens = tokenize(asr_sentence)
    corrected_tokens = []
    debug_rows = []

    for idx in range(len(tokens)):
        pred_info = predict_token_with_confidence(tokens, idx)
        original_token = pred_info["original_token"]

        apply_change = should_correct_token(pred_info, suspicious_tokens)

        if apply_change:
            final_token = pred_info["corrected_token"]
        else:
            final_token = original_token

        corrected_tokens.append(final_token)

        debug_rows.append({
            "index": idx,
            "original_token": original_token,
            "token_class": pred_info["token_class"],
            "pred_root": pred_info.get("pred_root"),
            "pred_suffix": pred_info.get("pred_suffix"),
            "root_conf": pred_info.get("root_conf"),
            "suffix_conf": pred_info.get("suffix_conf"),
            "model_corrected_token": pred_info.get("corrected_token"),
            "apply_change": apply_change,
            "final_token": final_token
        })

    corrected_sentence = " ".join(corrected_tokens)
    return corrected_sentence, debug_rows

In [80]:
SAFE_XLMR_RESULTS_CSV = "/content/drive/MyDrive/asr_xlmr_safe_results_sample_1000.csv"
SAFE_METRICS_JSON = "/content/drive/MyDrive/asr_xlmr_safe_metrics_sample_1000.json"

safe_rows = []

for idx, row in tqdm(asr_df.iterrows(), total=len(asr_df), desc="Running SAFE XLM-R correction"):
    row_id = row["id"]
    expected_text = str(row["expected_text"]).strip()
    generated_text = str(row["generated_text"]).strip()
    audio_path = str(row["audio_wav_path"]).strip()
    duration = row["duration"]
    error_msg = row["error"]

    if error_msg is not None and str(error_msg).strip() != "":
        xlmr_safe_text = ""
    else:
        xlmr_safe_text, _ = correct_sentence_with_xlmr_safe(generated_text, suspicious_tokens)

    safe_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "xlmr_safe_text": xlmr_safe_text,
        "error": error_msg
    })

safe_df = pd.DataFrame(safe_rows)
safe_df.to_csv(SAFE_XLMR_RESULTS_CSV, index=False)

print("Saved safe XLM-R results to:", SAFE_XLMR_RESULTS_CSV)
safe_df.head()

Running SAFE XLM-R correction:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved safe XLM-R results to: /content/drive/MyDrive/asr_xlmr_safe_results_sample_1000.csv


,id,audio_wav_path,duration,expected_text,generated_text,xlmr_safe_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,"ரெண்டு table book பண்ணுங்க please, ஜன்னல் பக்க...",Render table book பண்ணுங்க please. ஜன்னல் பக்க...,Render table book பண்ணுங்க please ஜன்னல் பக்கம...,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy Help Center site-ல Selling on Etsy-ல Paym...,ITZY help center site-ல selling on ITZY-ல paym...,ITZY help center site-ல selling on ITZY-ல paym...,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,None


In [82]:
from jiwer import wer, cer
import json

valid_safe_df = safe_df[safe_df["error"].isna() | (safe_df["error"] == "")].copy()

expected_list = valid_safe_df["expected_text"].astype(str).tolist()
asr_list = valid_safe_df["generated_text"].astype(str).tolist()
safe_list = valid_safe_df["xlmr_safe_text"].astype(str).tolist()

safe_metrics = {
    "num_rows_total": int(len(safe_df)),
    "num_rows_valid": int(len(valid_safe_df)),
    "asr_wer": float(wer(expected_list, asr_list)),
    "asr_cer": float(cer(expected_list, asr_list)),
    "xlmr_safe_wer": float(wer(expected_list, safe_list)),
    "xlmr_safe_cer": float(cer(expected_list, safe_list)),
}

print(json.dumps(safe_metrics, indent=2, ensure_ascii=False))

with open(SAFE_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(safe_metrics, f, indent=2, ensure_ascii=False)

print("Saved safe metrics to:", SAFE_METRICS_JSON)

{
  "num_rows_total": 1000,
  "num_rows_valid": 1000,
  "asr_wer": 0.2708534233365477,
  "asr_cer": 0.07405905804446246,
  "xlmr_safe_wer": 0.26530858244937316,
  "xlmr_safe_cer": 0.07909318197175484
}
Saved safe metrics to: /content/drive/MyDrive/asr_xlmr_safe_metrics_sample_1000.json


In [ ]:
DEBUG_TOKEN_ROWS_CSV = "/content/drive/MyDrive/asr_xlmr_safe_debug_token_rows.csv"

all_debug_rows = []

for i in range(min(100, len(asr_df))):  # only first 100 rows for debug
    row = asr_df.iloc[i]
    generated_text = str(row["generated_text"]).strip()
    error_msg = row["error"]

    if error_msg is not None and str(error_msg).strip() != "":
        continue

    _, debug_rows = correct_sentence_with_xlmr_safe(generated_text, suspicious_tokens)

    for d in debug_rows:
        d["row_id"] = row["id"]
        d["generated_text"] = generated_text
        all_debug_rows.append(d)

debug_df = pd.DataFrame(all_debug_rows)
debug_df.to_csv(DEBUG_TOKEN_ROWS_CSV, index=False)

print("Saved debug token rows to:", DEBUG_TOKEN_ROWS_CSV)
debug_df.head(20)

# MT5 Load

In [83]:
MT5_MODEL_PATH = "/content/drive/MyDrive/mt5_merged_model"

In [84]:
SAFE_XLMR_RESULTS_CSV = "/content/drive/MyDrive/asr_xlmr_safe_results_sample_1000.csv"

In [85]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MT5_MODEL_PATH = "/content/drive/MyDrive/mt5_merged_model"
device = "cuda" if torch.cuda.is_available() else "cpu"

mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL_PATH)

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    MT5_MODEL_PATH,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

mt5_model.to(device)
mt5_model.eval()

print("Loaded merged mT5 model.")

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

Loaded merged mT5 model.


In [86]:
import pandas as pd

SAFE_XLMR_RESULTS_CSV = "/content/drive/MyDrive/asr_xlmr_safe_results_sample_1000.csv"
FINAL_RESULTS_CSV = "/content/drive/MyDrive/asr_xlmr_mt5_results_sample_1000.csv"
FINAL_METRICS_JSON = "/content/drive/MyDrive/asr_xlmr_mt5_metrics_sample_1000.json"

safe_df = pd.read_csv(SAFE_XLMR_RESULTS_CSV)

print("Columns:", safe_df.columns.tolist())
print("Rows:", len(safe_df))

safe_df.head()

Columns: ['id', 'audio_wav_path', 'duration', 'expected_text', 'generated_text', 'xlmr_safe_text', 'error']
Rows: 1000


,id,audio_wav_path,duration,expected_text,generated_text,xlmr_safe_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,NaN
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,"ரெண்டு table book பண்ணுங்க please, ஜன்னல் பக்க...",Render table book பண்ணுங்க please. ஜன்னல் பக்க...,Render table book பண்ணுங்க please ஜன்னல் பக்கம...,NaN
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy Help Center site-ல Selling on Etsy-ல Paym...,ITZY help center site-ல selling on ITZY-ல paym...,ITZY help center site-ல selling on ITZY-ல paym...,NaN
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,NaN
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,NaN


In [87]:
def run_mt5_inference(input_text, max_input_len=256, max_new_tokens=256):
    try:
        prompt = "fix tamil-english: " + str(input_text).strip()

        inputs = mt5_tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_input_len
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        with torch.no_grad():
            outputs = mt5_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                early_stopping=True
            )

        text = mt5_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        return text, None

    except Exception as e:
        return "", str(e)

In [88]:
test_text = safe_df.loc[0, "xlmr_safe_text"]
print("INPUT :", test_text)

pred_text, err = run_mt5_inference(test_text)

print("OUTPUT:", pred_text)
print("ERROR :", err)

INPUT : Online class join பண்ண late ஆனதால start miss ஆயிட்டு
OUTPUT: Online class join பண்ண late ஆனதால start miss ஆயிட்டு.
ERROR : None


In [90]:
from tqdm.auto import tqdm
import pandas as pd

final_rows = []

for idx, row in tqdm(safe_df.iterrows(), total=len(safe_df), desc="Running mT5 on SAFE XLM-R output"):
    row_id = row["id"]
    audio_path = row["audio_wav_path"]
    duration = row["duration"]
    expected_text = str(row["expected_text"]).strip()
    generated_text = str(row["generated_text"]).strip()
    xlmr_safe_text = str(row["xlmr_safe_text"]).strip()
    error_msg = row["error"]

    has_error = (not pd.isna(error_msg)) and (str(error_msg).strip() != "")

    if has_error:
        mt5_text = ""
        mt5_error = error_msg
    else:
        mt5_text, mt5_error = run_mt5_inference(xlmr_safe_text)

    final_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "xlmr_safe_text": xlmr_safe_text,
        "mt5_final_text": mt5_text,
        "error": mt5_error
    })

final_df = pd.DataFrame(final_rows)
final_df.to_csv(FINAL_RESULTS_CSV, index=False)

print("Saved final pipeline results to:", FINAL_RESULTS_CSV)
final_df.head()

Running mT5 on SAFE XLM-R output:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved final pipeline results to: /content/drive/MyDrive/asr_xlmr_mt5_results_sample_1000.csv


,id,audio_wav_path,duration,expected_text,generated_text,xlmr_safe_text,mt5_final_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,"ரெண்டு table book பண்ணுங்க please, ஜன்னல் பக்க...",Render table book பண்ணுங்க please. ஜன்னல் பக்க...,Render table book பண்ணுங்க please ஜன்னல் பக்கம...,Render table book பண்ணுங்க please ஜன்னல் பக்கம...,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy Help Center site-ல Selling on Etsy-ல Paym...,ITZY help center site-ல selling on ITZY-ல paym...,ITZY help center site-ல selling on ITZY-ல paym...,ITZY help center site-ல selling on ITZY-ல paym...,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,None


In [91]:
from jiwer import wer, cer
import json

valid_final_df = final_df[final_df["error"].isna() | (final_df["error"] == "")].copy()

expected_list = valid_final_df["expected_text"].astype(str).tolist()
asr_list = valid_final_df["generated_text"].astype(str).tolist()
xlmr_list = valid_final_df["xlmr_safe_text"].astype(str).tolist()
mt5_list = valid_final_df["mt5_final_text"].astype(str).tolist()

final_metrics = {
    "num_rows_total": int(len(final_df)),
    "num_rows_valid": int(len(valid_final_df)),
    "asr_wer": float(wer(expected_list, asr_list)),
    "asr_cer": float(cer(expected_list, asr_list)),
    "xlmr_safe_wer": float(wer(expected_list, xlmr_list)),
    "xlmr_safe_cer": float(cer(expected_list, xlmr_list)),
    "xlmr_mt5_wer": float(wer(expected_list, mt5_list)),
    "xlmr_mt5_cer": float(cer(expected_list, mt5_list)),
}

print(json.dumps(final_metrics, indent=2, ensure_ascii=False))

with open(FINAL_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=2, ensure_ascii=False)

print("Saved final metrics to:", FINAL_METRICS_JSON)

{
  "num_rows_total": 1000,
  "num_rows_valid": 1000,
  "asr_wer": 0.2708534233365477,
  "asr_cer": 0.07405905804446246,
  "xlmr_safe_wer": 0.26530858244937316,
  "xlmr_safe_cer": 0.07909318197175484,
  "xlmr_mt5_wer": 0.24795081967213115,
  "xlmr_mt5_cer": 0.07694776674099602
}
Saved final metrics to: /content/drive/MyDrive/asr_xlmr_mt5_metrics_sample_1000.json


# ASR --> mT5

In [92]:
from tqdm.auto import tqdm
import pandas as pd

MT5_BASELINE_RESULTS_CSV = "/content/drive/MyDrive/asr_mt5_results_sample_1000.csv"

baseline_rows = []

for idx, row in tqdm(safe_df.iterrows(), total=len(safe_df), desc="Running mT5 on ASR output"):
    row_id = row["id"]
    audio_path = row["audio_wav_path"]
    duration = row["duration"]
    expected_text = str(row["expected_text"]).strip()
    generated_text = str(row["generated_text"]).strip()
    error_msg = row["error"]

    # correct NaN handling
    has_error = (not pd.isna(error_msg)) and (str(error_msg).strip() != "")

    if has_error:
        mt5_text = ""
        mt5_error = error_msg
    else:
        mt5_text, mt5_error = run_mt5_inference(generated_text)

    baseline_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "mt5_from_asr_text": mt5_text,
        "error": mt5_error
    })

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(MT5_BASELINE_RESULTS_CSV, index=False)

print("Saved ASR → mT5 results to:", MT5_BASELINE_RESULTS_CSV)
baseline_df.head()

Running mT5 on ASR output:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved ASR → mT5 results to: /content/drive/MyDrive/asr_mt5_results_sample_1000.csv


,id,audio_wav_path,duration,expected_text,generated_text,mt5_from_asr_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,"ரெண்டு table book பண்ணுங்க please, ஜன்னல் பக்க...",Render table book பண்ணுங்க please. ஜன்னல் பக்க...,Render table book பண்ணுங்க please ஜன்னல் பக்கம...,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy Help Center site-ல Selling on Etsy-ல Paym...,ITZY help center site-ல selling on ITZY-ல paym...,ITZY help center site-ல selling on ITZY-ல paym...,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,None


In [93]:
from jiwer import wer, cer
import json

MT5_BASELINE_METRICS_JSON = "/content/drive/MyDrive/asr_mt5_metrics_sample_1000.json"

valid_baseline_df = baseline_df[
    baseline_df["error"].isna() | (baseline_df["error"] == "")
].copy()

expected_list = valid_baseline_df["expected_text"].astype(str).tolist()
asr_list = valid_baseline_df["generated_text"].astype(str).tolist()
mt5_list = valid_baseline_df["mt5_from_asr_text"].astype(str).tolist()

baseline_metrics = {
    "num_rows_total": int(len(baseline_df)),
    "num_rows_valid": int(len(valid_baseline_df)),
    "asr_wer": float(wer(expected_list, asr_list)),
    "asr_cer": float(cer(expected_list, asr_list)),
    "mt5_from_asr_wer": float(wer(expected_list, mt5_list)),
    "mt5_from_asr_cer": float(cer(expected_list, mt5_list)),
}

print(json.dumps(baseline_metrics, indent=2, ensure_ascii=False))

with open(MT5_BASELINE_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=2, ensure_ascii=False)

print("Saved baseline metrics to:", MT5_BASELINE_METRICS_JSON)

{
  "num_rows_total": 1000,
  "num_rows_valid": 1000,
  "asr_wer": 0.2708534233365477,
  "asr_cer": 0.07405905804446246,
  "mt5_from_asr_wer": 0.24795081967213115,
  "mt5_from_asr_cer": 0.07068045138185013
}
Saved baseline metrics to: /content/drive/MyDrive/asr_mt5_metrics_sample_1000.json


In [94]:
FINAL_COMPARISON_JSON = "/content/drive/MyDrive/final_pipeline_comparison.json"

comparison_metrics = {
    "ASR": {
        "WER": float(wer(expected_list, asr_list)),
        "CER": float(cer(expected_list, asr_list)),
    },
    "ASR_mT5": {
        "WER": float(wer(expected_list, mt5_list)),
        "CER": float(cer(expected_list, mt5_list)),
    },
    "ASR_XLMR": {
        "WER": float(wer(expected_list, safe_df["xlmr_safe_text"].astype(str).tolist())),
        "CER": float(cer(expected_list, safe_df["xlmr_safe_text"].astype(str).tolist())),
    },
    "ASR_XLMR_mT5": {
        "WER": float(wer(expected_list, final_df["mt5_final_text"].astype(str).tolist())),
        "CER": float(cer(expected_list, final_df["mt5_final_text"].astype(str).tolist())),
    }
}

print(json.dumps(comparison_metrics, indent=2, ensure_ascii=False))

with open(FINAL_COMPARISON_JSON, "w", encoding="utf-8") as f:
    json.dump(comparison_metrics, f, indent=2, ensure_ascii=False)

print("Saved final comparison to:", FINAL_COMPARISON_JSON)

{
  "ASR": {
    "WER": 0.2708534233365477,
    "CER": 0.07405905804446246
  },
  "ASR_mT5": {
    "WER": 0.24795081967213115,
    "CER": 0.07068045138185013
  },
  "ASR_XLMR": {
    "WER": 0.26530858244937316,
    "CER": 0.07909318197175484
  },
  "ASR_XLMR_mT5": {
    "WER": 0.24795081967213115,
    "CER": 0.07694776674099602
  }
}
Saved final comparison to: /content/drive/MyDrive/final_pipeline_comparison.json
